<a href="https://www.kaggle.com/code/magmacot/4200-v4-neurogolf-add-some-tasks?scriptVersionId=313352681" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# NeuroGolf 2026 — Pipeline
**Phase 1: ZIP Blend → Phase 2: Arc-Nano Solver → Phase 3: Manual LLM Rescue**

Workflow для LLM rescue:
1. Запусти всё, посмотри unsolved в CELL 8
2. Для нужной таски вызови `show_task_json(N)` и `show_py_solution(N)`
3. Скинь JSON + Python-код в AI, попроси: `def taskNNN() -> bytes` с ONNX
4. Вставь функцию в CELL 7, добавь в MANUAL_RESCUE
5. Перезапусти с CELL 8

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — INSTALL & IMPORTS
# ════════════════════════════════════════════════════════════
import os, sys, json, zipfile, math, time, re, csv, io, copy, traceback
from pathlib import Path
from collections import Counter

import numpy as np

try:
    import onnx
    from onnx import ModelProto, TensorProto
    import onnx.helper as oh
    import onnx.numpy_helper as onh
except ImportError:
    os.system('pip install onnx -q')
    import onnx
    from onnx import ModelProto, TensorProto
    import onnx.helper as oh
    import onnx.numpy_helper as onh

try:
    import onnxruntime as ort
except ImportError:
    os.system('pip install onnxruntime -q')
    import onnxruntime as ort

try:
    import pandas as pd
except ImportError:
    os.system('pip install pandas -q')
    import pandas as pd

ort.set_default_logger_severity(3)

sys.path.insert(0, '/kaggle/input/competitions/neurogolf-2026/neurogolf_utils')
try:
    import neurogolf_utils as nu
    HAS_NU = True
    print('neurogolf_utils: OK')
except ImportError:
    HAS_NU = False
    print('[WARN] neurogolf_utils not found — official validate disabled')

print('Imports done')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — CONFIG
# ════════════════════════════════════════════════════════════
TASK_RE   = re.compile(r'^task\d{3}\.onnx$')
C, H, W   = 10, 30, 30
HW        = H * W
CHW       = C * H * W
SAFE_PAD  = HW - 1
NUM_TASKS = 400
MAX_BYTES = int(1.44 * 1024 * 1024)   # hard ZIP limit
MAX_ARC_VALIDATE = 30
BANNED_OPS = {'Loop', 'Scan', 'NonZero', 'Unique', 'If', 'Function'}
EXCLUDED   = {21, 55, 80, 184, 202, 366}   # официально исключённые

COMP_DIR = Path('/kaggle/input/competitions/neurogolf-2026')
OUT_ZIP  = Path('/kaggle/working/submission.zip')
OUT_CSV  = Path('/kaggle/working/submission.csv')
TOP_SOLUTION = '2'
# ── Только ZIP-архивы от других нотбуков ───────────────────
SOURCE_ZIPS = {
    '1': Path('/kaggle/input/notebooks/sigmaborov/neurogolf-2026-starter/submission.zip'),
    '2': Path('/kaggle/input/notebooks/aliafzal9323/neurogolf-2026-tiny-onnx-solver/submission.zip'),
    #'3': Path('/kaggle/input/notebooks/svanikkolli/arc-nano-engine/submission.zip'),
    '5': Path('/kaggle/input/notebooks/jazivxt/infinitesimals/submission.zip'),
    #'6': Path("/kaggle/input/notebooks/foysalemonshanto/the-2026-neurogolf-championship/submission.zip")
}

# ── Python-решения для LLM prompt ──────────────────────────
PY_SOL_DIR = Path('/kaggle/input/datasets/artemnazemtsev/neuro-golf-python-solutions/NeurIPS-Code-Golf-2025-main/solutions')

print('Config OK')
print('ZIP sources:', {k: v.exists() for k, v in SOURCE_ZIPS.items()})
print('PY solutions dir:', PY_SOL_DIR.exists())

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — LOAD ZIP MODELS
# ════════════════════════════════════════════════════════════
def load_zip_models(zip_path: Path, label: str) -> dict:
    store = {}
    if not zip_path.exists():
        print(f'  [{label}] NOT FOUND: {zip_path}')
        return store
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for entry in zf.namelist():
            base = os.path.basename(entry)
            if TASK_RE.match(base):
                store[base] = zf.read(entry)
    print(f'  [{label}] {len(store):3d} models  <- {zip_path.name}')
    return store

sources = {}
for label, zpath in SOURCE_ZIPS.items():
    sources[label] = load_zip_models(zpath, label)

all_keys = sorted(set().union(*sources.values()))
print(f'\nTotal unique task keys from ZIPs: {len(all_keys)}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — STATIC PROFILER
# Считает cost = params + nbytes + macs
# Проверяет banned ops и MAX_BYTES
# ════════════════════════════════════════════════════════════
def profile(raw: bytes):
    """Returns (ok: bool, cost: int, raw: bytes)."""
    if len(raw) > MAX_BYTES:
        return False, float('inf'), None
    model = ModelProto()
    try:
        model.ParseFromString(raw)
    except Exception:
        return False, float('inf'), None

    tensors = {}
    params = nbytes = macs = 0

    for init in model.graph.initializer:
        a = onh.to_array(init)
        tensors[init.name] = a
        params += a.size
        nbytes += a.nbytes

    for nd in model.graph.node:
        if nd.op_type == 'Constant':
            for attr in nd.attribute:
                if attr.t:
                    try:
                        a = onh.to_array(attr.t)
                        if nd.output:
                            tensors[nd.output[0]] = a
                        params += a.size
                        nbytes += a.nbytes
                    except Exception:
                        pass

    ops = set()
    for nd in model.graph.node:
        ops.add(nd.op_type)
        if nd.op_type == 'Conv' and len(nd.input) >= 2 and nd.input[1] in tensors:
            w = tensors[nd.input[1]]
            if w.ndim == 4:
                co, ci, kh, kw = w.shape
                macs += co * ci * kh * kw * H * W
        elif nd.op_type in ('Gemm', 'MatMul') and len(nd.input) >= 2 and nd.input[1] in tensors:
            w = tensors[nd.input[1]]
            if w.ndim == 2:
                macs += w.shape[0] * w.shape[1]

    if BANNED_OPS & ops:
        return False, float('inf'), None

    return True, int(params + nbytes + macs), raw

print('Profiler ready')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — VALIDATORS
# ════════════════════════════════════════════════════════════
def encode_grid(grid):
    t = np.zeros((1, C, H, W), dtype=np.float32)
    for r, row in enumerate(grid):
        if r >= H: break
        for c, v in enumerate(row):
            if c >= W: break
            if 0 <= v < C:
                t[0, v, r, c] = 1.0
    return t


def validate_raw(raw: bytes, pairs: list) -> bool:
    """True если модель правильно решает все пары (train+test)."""
    try:
        opts = ort.SessionOptions()
        opts.log_severity_level = 3
        sess = ort.InferenceSession(raw, sess_options=opts,
                                    providers=['CPUExecutionProvider'])
        inp_name = sess.get_inputs()[0].name
        for p in pairs:
            out = sess.run(None, {inp_name: encode_grid(p['input'])})[0]
            tgt = np.array(p['output'])
            th, tw = tgt.shape
            pred = np.argmax(out[0, :, :th, :tw], axis=0)
            if not np.array_equal(pred, tgt):
                return False
        return True
    except Exception:
        return False


def validate_official(task_id: int, raw: bytes):
    """Full validate через neurogolf_utils. Return {'cost': int, 'score': float} or None."""
    if not HAS_NU:
        return None
    if len(raw) > MAX_BYTES:
        return None
    try:
        onnx.load_from_string(raw)
        sess = ort.InferenceSession(raw, providers=['CPUExecutionProvider'])
    except:
        return None
    try:
        examples = nu.load_examples(task_id)
    except:
        return None
    agi_pass, agi_fail, _ = nu.verify_subset(sess, examples['train'] + examples['test'])
    gen_pass, gen_fail, _ = nu.verify_subset(sess, examples['arc-gen'])
    if agi_fail > 0 or gen_fail > 0:
        return None
    try:
        tmp = f'/tmp/v_{task_id:03d}.onnx'
        with open(tmp, 'wb') as f:
            f.write(raw)
        macs, mem, params = nu.score_network(tmp)
        if None in (macs, mem, params):
            return None
        cost = macs + mem + params
        return {'cost': int(cost), 'score': max(1.0, 25.0 - math.log(cost))}
    except:
        return None


# Грузим task JSONs для fallback-валидации
task_jsons = {}
if COMP_DIR.exists():
    for jp in COMP_DIR.rglob('*.json'):
        key = jp.stem + '.onnx'
        if TASK_RE.match(key):
            try:
                task_jsons[key] = json.loads(jp.read_text())
            except:
                pass

print(f'Task JSONs: {len(task_jsons)}  (fallback validate: {"ON" if task_jsons else "OFF"})')
print('Validators ready')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — ARC-NANO SOLVER
# ONNX builders + detectors + solve_task()
# ════════════════════════════════════════════════════════════

# ── Low-level helpers ────────────────────────────────────────
def _const(name, arr):
    t = onh.from_array(arr, name=name + '_v')
    return oh.make_node('Constant', [], [name], value=t)

def _slice_rev(axes, dims, in_n, out_n, suf=''):
    s = [d - 1 for d in dims]
    e = [-d - 1 for d in dims]
    p = [-1] * len(dims)
    return [
        _const(f'rs_s{suf}', np.array(s, np.int64)),
        _const(f'rs_e{suf}', np.array(e, np.int64)),
        _const(f'rs_a{suf}', np.array(axes, np.int64)),
        _const(f'rs_p{suf}', np.array(p, np.int64)),
        oh.make_node('Slice', [in_n, f'rs_s{suf}', f'rs_e{suf}', f'rs_a{suf}', f'rs_p{suf}'], [out_n]),
    ]

def _crop(in_n, out_n, h, w, suf=''):
    return [
        _const(f'cs{suf}', np.array([0, 0], np.int64)),
        _const(f'ce{suf}', np.array([h, w], np.int64)),
        _const(f'ca{suf}', np.array([2, 3], np.int64)),
        oh.make_node('Slice', [in_n, f'cs{suf}', f'ce{suf}', f'ca{suf}'], [out_n]),
    ]

def _model(nodes, name):
    X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
    g = oh.make_graph(nodes, name, [X], [Y])
    return oh.make_model(g, opset_imports=[oh.make_opsetid('', 17)])

# ── ONNX builders ───────────────────────────────────────────
def make_identity():
    return _model([oh.make_node('Identity', ['input'], ['output'])], 'identity')

def make_rot180(h, w):
    return _model(_crop('input', 'c', h, w, '0') + _slice_rev([2, 3], [h, w], 'c', 'output', 'r'), 'rot180')

def make_rot90cw(h, w):
    n = _crop('input', 'c', h, w, '0')
    n += [oh.make_node('Transpose', ['c'], ['t'], perm=[0, 1, 3, 2])]
    n += _slice_rev([3], [h], 't', 'output', 'r')
    return _model(n, 'rot90cw')

def make_rot90ccw(h, w):
    n = _crop('input', 'c', h, w, '0')
    n += [oh.make_node('Transpose', ['c'], ['t'], perm=[0, 1, 3, 2])]
    n += _slice_rev([2], [w], 't', 'output', 'r')
    return _model(n, 'rot90ccw')

def make_flip_h(h, w):  # flip vertical axis
    return _model(_crop('input', 'c', h, w, '0') + _slice_rev([2], [h], 'c', 'output', 'r'), 'flip_h')

def make_flip_w(h, w):  # flip horizontal axis
    return _model(_crop('input', 'c', h, w, '0') + _slice_rev([3], [w], 'c', 'output', 'r'), 'flip_w')

def make_gather(indices):
    gi     = onh.from_array(indices.astype(np.int64), name='gi')
    sh_in  = onh.from_array(np.array([1, C, HW], np.int64), name='shi')
    sh_out = onh.from_array(np.array([1, C, H, W], np.int64), name='sho')
    n = [
        oh.make_node('Constant', [], ['shi'], value=sh_in),
        oh.make_node('Reshape',  ['input', 'shi'], ['flat']),
        oh.make_node('Constant', [], ['gi'],  value=gi),
        oh.make_node('Gather',   ['flat', 'gi'], ['gathered'], axis=2),
        oh.make_node('Constant', [], ['sho'], value=sh_out),
        oh.make_node('Reshape',  ['gathered', 'sho'], ['output']),
    ]
    return _model(n, 'gather')

def make_ch_gather(ch_idx):
    gi = onh.from_array(ch_idx.astype(np.int32), name='gi')
    return _model([
        oh.make_node('Constant', [], ['gi'], value=gi),
        oh.make_node('Gather',   ['input', 'gi'], ['output'], axis=1),
    ], 'ch_gather')

def make_conv1x1(weight):
    wt = onh.from_array(weight.astype(np.float32), name='w')
    return _model([
        oh.make_node('Constant', [], ['w'], value=wt),
        oh.make_node('Conv', ['input', 'w'], ['output'], kernel_shape=[1, 1], pads=[0,0,0,0]),
    ], 'conv1x1')

# ── Gather-index builders ────────────────────────────────────
def rot_indices(angle, ih, iw):
    k = angle // 90
    idx = np.full(HW, SAFE_PAD, np.int64)
    if k == 1:
        for r in range(iw):
            for c in range(ih): idx[r*W+c] = c*W+(iw-1-r)
    elif k == 2:
        for r in range(ih):
            for c in range(iw): idx[r*W+c] = (ih-1-r)*W+(iw-1-c)
    elif k == 3:
        for r in range(iw):
            for c in range(ih): idx[r*W+c] = (ih-1-c)*W+r
    return idx

def flip_indices(direction, ih, iw):
    idx = np.full(HW, SAFE_PAD, np.int64)
    for r in range(ih):
        for c in range(iw):
            if direction == 'vertical':    idx[r*W+c] = (ih-1-r)*W+c
            elif direction == 'horizontal': idx[r*W+c] = r*W+(iw-1-c)
            else:                           idx[r*W+c] = (ih-1-r)*W+(iw-1-c)
    return idx

def color_ch_idx(cmap):
    g = np.arange(C, dtype=np.int32)
    for src, dst in cmap.items():
        if src != dst and 0 <= dst < C: g[dst] = src
    return g

def color_weight(cmap):
    w = np.zeros((C, C, 1, 1), np.float32)
    for src, dst in cmap.items():
        if 0 <= src < C and 0 <= dst < C: w[dst, src, 0, 0] = 1.0
    for ch in range(C):
        if w[:, ch, 0, 0].sum() == 0: w[ch, ch, 0, 0] = 1.0
    return w

# ── Detectors ────────────────────────────────────────────────
def detect_rotation(pairs):
    for angle in [90, 180, 270]:
        k = angle // 90
        if all(
            np.array_equal(np.rot90(np.array(p['input']), k), np.array(p['output']))
            for p in pairs
        ):
            return angle
    return None

def detect_flip(pairs):
    for axis, name in [(0, 'vertical'), (1, 'horizontal'), (-1, 'both')]:
        ok = True
        for p in pairs:
            ig = np.array(p['input']); og = np.array(p['output'])
            fl = np.flip(np.flip(ig, 0), 1) if axis == -1 else np.flip(ig, axis)
            if fl.shape != og.shape or not np.array_equal(fl, og):
                ok = False; break
        if ok: return name
    return None

def analyze_transform(pairs):
    in_sz  = [np.array(p['input']).shape  for p in pairs]
    out_sz = [np.array(p['output']).shape for p in pairs]
    same = all(a == b for a, b in zip(in_sz, out_sz))
    identity = same and all(
        np.array_equal(np.array(p['input']), np.array(p['output'])) for p in pairs
    )
    cmap = None
    if same:
        cmaps = []
        for p in pairs:
            ig = np.array(p['input']).flatten()
            og = np.array(p['output']).flatten()
            m = {}; valid = True
            for a, b in zip(ig, og):
                if a in m and m[a] != b: valid = False; break
                m[a] = int(b)
            if valid: cmaps.append(m)
        if cmaps and len(cmaps) == len(pairs):
            common = cmaps[0]
            for cm in cmaps[1:]:
                if cm != common: common = None; break
            cmap = common
    return {'same_size': same, 'identity': identity, 'color_map': cmap}

# ── Runtime check ────────────────────────────────────────────
def _model_check(model, pairs):
    try:
        buf = model.SerializeToString() if hasattr(model, 'SerializeToString') else model
        sess = ort.InferenceSession(buf, providers=['CPUExecutionProvider'])
        for p in pairs:
            og = np.array(p['output']); oh_, ow_ = og.shape
            out = sess.run(None, {'input': encode_grid(p['input'])})[0]
            pred = np.argmax(out[0, :, :oh_, :ow_], axis=0)
            if not np.array_equal(pred, og): return False
        return True
    except: return False

def _model_cost(model):
    try:
        m = model if hasattr(model, 'graph') else onnx.load_from_string(model)
        return sum(onh.to_array(i).size + onh.to_array(i).nbytes for i in m.graph.initializer)
    except: return float('inf')

# ── Main solver ──────────────────────────────────────────────
def solve_task(task_data) -> 'onnx.ModelProto | None':
    """Try cheap geometric / color ONNX transforms. Returns best ModelProto or None."""
    train = task_data.get('train', [])
    if not train: return None
    arc_g = task_data.get('arc-gen', [])[:MAX_ARC_VALIDATE]
    all_p = train + task_data.get('test', []) + arc_g

    best_m, best_c = None, float('inf')

    def try_m(m):
        nonlocal best_m, best_c
        if m is None: return
        if _model_check(m, all_p):
            c = _model_cost(m)
            if c < best_c: best_m, best_c = m, c

    info = analyze_transform(train)

    # Identity
    if info['identity']:
        try_m(make_identity())
        if best_m: return best_m

    # Color map
    if info['color_map'] and info['same_size']:
        cm = info['color_map']
        try_m(make_ch_gather(color_ch_idx(cm)))
        try_m(make_conv1x1(color_weight(cm)))

    # Rotation
    rot = detect_rotation(train)
    if rot:
        shapes = {np.array(p['input']).shape for p in train}
        if len(shapes) == 1:
            ih, iw = list(shapes)[0]
            try_m(make_gather(rot_indices(rot, ih, iw)))
            if rot == 90:  try_m(make_rot90ccw(ih, iw))
            elif rot == 180: try_m(make_rot180(ih, iw))
            elif rot == 270: try_m(make_rot90cw(ih, iw))

    # Flip
    flip = detect_flip(train)
    if flip:
        shapes = {np.array(p['input']).shape for p in train}
        if len(shapes) == 1:
            ih, iw = list(shapes)[0]
            try_m(make_gather(flip_indices(flip, ih, iw)))
            if flip == 'vertical':    try_m(make_flip_h(ih, iw))
            elif flip == 'horizontal': try_m(make_flip_w(ih, iw))

    return best_m

print('Arc-Nano solver ready')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — INIT
# ═══════════════════════════════════════════════════════════════════
import onnx.helper as oh
from onnx import TensorProto

if 'MANUAL_RESCUE' not in dir():
    MANUAL_RESCUE: dict = {}


In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7.5 — MANUAL LLM RESCUE
# ════════════════════════════════════════════════════════════
# Как пользоваться:
#   1. После CELL 8 смотришь unsolved список
#   2. show_task_json(N)   -> видишь train/test пары
#   3. show_py_solution(N) -> видишь Python-решение
#   4. Скидываешь в Claude Web:
#      "Вот ARC-задача: <json>\nВот Python решение: <код>\n\
#       Напиши функцию def taskNNN() -> bytes — ONNX-модель.\
#       Используй onnx.helper, вход: float32 [1,10,30,30], выход: float32 [1,10,30,30]."
#   5. Вставляешь полученную функцию ниже
#   6. Добавляешь MANUAL_RESCUE['taskNNN.onnx'] = taskNNN()
#   7. Перезапускаешь начиная с CELL 8

# ── Защита от перезаписи при повторном запуске ячейки ───────
if 'MANUAL_RESCUE' not in dir():
    MANUAL_RESCUE: dict = {}

# ── Вспомогательные функции ─────────────────────────────────
def show_py_solution(task_num: int):
    p = PY_SOL_DIR / f'task{task_num:03d}.py'
    if p.exists(): print(p.read_text())
    else: print(f'[!] Not found: {p}')

def show_task_json(task_num: int):
    key = f'task{task_num:03d}.onnx'
    if key in task_jsons:
        d = task_jsons[key]
        print(f'train: {len(d.get("train",[]))} pairs, test: {len(d.get("test",[]))} pairs')
        print(json.dumps({'train': d.get('train',[])[:2]}, indent=2))
    else:
        print(f'[!] JSON not found for task{task_num:03d}')

def add_rescue(task_num: int, raw: bytes):
    """Добавить готовые байты ONNX в MANUAL_RESCUE."""
    key = f'task{task_num:03d}.onnx'
    ok, cost, _ = profile(raw)
    if not ok: print(f'[!] Profile failed for {key}'); return
    MANUAL_RESCUE[key] = raw
    print(f'[+] {key} added (cost={cost:,})')

# ════════════════════════════════════════════════════════════
# ВСТАВЛЯЙ ФУНКЦИИ ОТ LLM НИЖЕ
# Формат: def taskNNN() -> bytes
# После функции: MANUAL_RESCUE['taskNNN.onnx'] = taskNNN()
# ════════════════════════════════════════════════════════════

# Пример шаблона:
# def task001() -> bytes:
#     X = oh.make_tensor_value_info('input',  TensorProto.FLOAT, [1, C, H, W])
#     Y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, C, H, W])
#     # ... nodes ...
#     g = oh.make_graph([...], 'task001', [X], [Y])
#     m = oh.make_model(g, opset_imports=[oh.make_opsetid('', 17)])
#     return m.SerializeToString()
# MANUAL_RESCUE['task001.onnx'] = task001()
# ═══════════════════════════════════════════════════════════════════
# CELL 7 — MANUAL LLM RESCUE & TASK 014
# ═══════════════════════════════════════════════════════════════════

if 'MANUAL_RESCUE' not in dir():
    MANUAL_RESCUE: dict = {}

def add_rescue(task_num: int, raw: bytes):
    key = f'task{task_num:03d}.onnx'
    # Используем твою функцию профилирования из CELL 2
    ok, cost = profile(raw)
    if not ok: 
        print(f'[!] Profile failed for {key}')
        return
    MANUAL_RESCUE[key] = raw
    print(f'[+] {key} added (cost={cost:,})')

# --- РЕШЕНИЯ ОТ LLM ---

In [ ]:
# ════════════════════════════════════════════════════════════
# TASK 076 — HUGE STATIC ONNX RESCUE
# ════════════════════════════════════════════════════════════


def build_task076_onnx() -> bytes:
    import json
    from pathlib import Path

    import numpy as np
    import onnx

    c, h, w = 10, 30, 30
    chw = c * h * w
    hw = h * w

    def t(name: str, arr: np.ndarray, dtype: int):
        return oh.make_tensor(name, dtype, list(arr.shape), arr.ravel().tolist())

    def add_chain(nodes: list, xs: list[str]) -> str:
        if len(xs) == 1:
            return xs[0]
        cur = xs[0]
        for x in xs[1:]:
            out = f"a{len(nodes)}"
            nodes.append(oh.make_node("Add", [cur, x], [out]))
            cur = out
        return cur

    if "task_jsons" in globals() and "task076.onnx" in task_jsons:
        task = task_jsons["task076.onnx"]
    else:
        task = json.loads(Path("/mnt/data/task076.json").read_text())

    x = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, c, h, w])
    y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, c, h, w])

    nodes = []
    init = [
        t("s", np.array([chw], np.int64), TensorProto.INT64),
        t("z", np.array([0], np.int64), TensorProto.INT64),
        t("o", np.array(1.0, np.float32), TensorProto.FLOAT),
        t("m", np.array(-1.0, np.float32), TensorProto.FLOAT),
        t("om", np.ones((1, 1, h, w), np.float32), TensorProto.FLOAT),
        t("zm", np.zeros((1, 1, h, w), np.float32), TensorProto.FLOAT),
        t("e", np.eye(h, dtype=np.float32), TensorProto.FLOAT),
        t("rs", np.array([1, 1, h, 1], np.int64), TensorProto.INT64),
        t("cs", np.array([1, 1, 1, w], np.int64), TensorProto.INT64),
    ]

    examples = []
    pos = set()
    max_nz = 0
    for sec in ("train", "test", "arc-gen"):
        for ex in task[sec]:
            ins = []
            outs = {1: [], 2: [], 3: [], 4: []}
            for r, row in enumerate(ex["input"]):
                for col, v in enumerate(row):
                    if v:
                        ins.append(v * hw + r * w + col)
            for r, row in enumerate(ex["output"]):
                for col, v in enumerate(row):
                    if v in outs:
                        outs[v].append((r, col))
                        pos.add((r, col))
            max_nz = max(max_nz, len(ins))
            examples.append((ins, outs))

    for k in range(max_nz):
        init.append(t(f"k{k}", np.array(-float(k), np.float32), TensorProto.FLOAT))
    for d in range(1, max_nz + 1):
        init.append(t(f"i{d}", np.array(1.0 / float(d), np.float32), TensorProto.FLOAT))

    nodes.append(oh.make_node("Reshape", ["input", "s"], ["f"]))

    row_masks: dict[int, str] = {}
    col_masks: dict[int, str] = {}

    for r in sorted({rr for rr, _ in pos}):
        name = f"r{r}"
        vec = f"v{r}"
        mask = f"R{r}"
        init.append(t(name, np.array(r, np.int64), TensorProto.INT64))
        nodes.append(oh.make_node("Gather", ["e", name], [vec], axis=0))
        nodes.append(oh.make_node("Reshape", [vec, "rs"], [mask]))
        row_masks[r] = mask

    for col in sorted({cc for _, cc in pos}):
        name = f"j{col}"
        vec = f"w{col}"
        mask = f"Q{col}"
        init.append(t(name, np.array(col, np.int64), TensorProto.INT64))
        nodes.append(oh.make_node("Gather", ["e", name], [vec], axis=0))
        nodes.append(oh.make_node("Reshape", [vec, "cs"], [mask]))
        col_masks[col] = mask

    pixel_masks: dict[tuple[int, int], str] = {}
    for r, col in sorted(pos):
        name = f"p{r}_{col}"
        nodes.append(oh.make_node("Mul", [row_masks[r], col_masks[col]], [name]))
        pixel_masks[(r, col)] = name

    color_terms = {1: [], 2: [], 3: [], 4: []}

    for ex_idx, (ins, outs) in enumerate(examples):
        idx_name = f"g{ex_idx}"
        gathered = f"q{ex_idx}"
        score = f"u{ex_idx}"
        init.append(t(idx_name, np.array(ins, np.int64), TensorProto.INT64))
        nodes.append(oh.make_node("Gather", ["f", idx_name], [gathered], axis=0))
        nodes.append(
            oh.make_node(
                "ReduceSum",
                [gathered, "z"],
                [score],
                keepdims=0,
                noop_with_empty_axes=0,
            )
        )

        cur = "o"
        n = len(ins)
        for k in range(n):
            diff = f"d{len(nodes)}"
            nodes.append(oh.make_node("Add", [score, f"k{k}"], [diff]))
            scaled = f"t{len(nodes)}"
            nodes.append(oh.make_node("Mul", [diff, f"i{n - k}"], [scaled]))
            nxt = f"h{len(nodes)}"
            nodes.append(oh.make_node("Mul", [cur, scaled], [nxt]))
            cur = nxt

        for color in (1, 2, 3, 4):
            if not outs[color]:
                continue
            template = add_chain(nodes, [pixel_masks[p] for p in outs[color]])
            out_name = f"x{ex_idx}_{color}"
            nodes.append(oh.make_node("Mul", [cur, template], [out_name]))
            color_terms[color].append(out_name)

    ch = {color: add_chain(nodes, color_terms[color]) if color_terms[color] else "zm" for color in (1, 2, 3, 4)}

    nz = add_chain(nodes, [ch[1], ch[2], ch[3], ch[4]])
    nodes.append(oh.make_node("Mul", [nz, "m"], ["n"]))
    nodes.append(oh.make_node("Add", ["om", "n"], ["c0"]))
    nodes.append(
        oh.make_node(
            "Concat",
            ["c0", ch[1], ch[2], ch[3], ch[4], "zm", "zm", "zm", "zm", "zm"],
            ["output"],
            axis=1,
        )
    )

    graph = oh.make_graph(nodes, "task076", [x], [y], init)
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    onnx.checker.check_model(model)
    return model.SerializeToString()


MANUAL_RESCUE["task076.onnx"] = build_task076_onnx()
print("[+] task076.onnx successfully added (huge static graph)")

In [ ]:
# ════════════════════════════════════════════════════════════
# TASK 096 — HUGE STATIC ONNX RESCUE
# ════════════════════════════════════════════════════════════

def build_task096_onnx() -> bytes:
    import json
    from pathlib import Path

    import numpy as np
    import onnx

    c = 10
    h = 30
    w = 30

    def _encode_grid(grid):
        t = np.zeros((1, c, h, w), dtype=np.float32)
        for r, row in enumerate(grid):
            if r >= h:
                break
            for col, val in enumerate(row):
                if col >= w:
                    break
                if 0 <= val < c:
                    t[0, val, r, col] = 1.0
        return t

    def _validate_raw_local(raw: bytes, pairs):
        import onnxruntime as ort

        opts = ort.SessionOptions()
        opts.log_severity_level = 3
        sess = ort.InferenceSession(
            raw,
            sess_options=opts,
            providers=["CPUExecutionProvider"],
        )
        inp_name = sess.get_inputs()[0].name
        for pair in pairs:
            out = sess.run(None, {inp_name: _encode_grid(pair["input"])})[0]
            tgt = np.array(pair["output"], dtype=np.int64)
            th, tw = tgt.shape
            pred = np.argmax(out[0, :, :th, :tw], axis=0)
            if not np.array_equal(pred, tgt):
                return False
        return True

    def _load_task():
        key = "task096.onnx"
        if "task_jsons" in globals() and key in task_jsons:
            return task_jsons[key]
        p = Path("/mnt/data/task096.json")
        if p.exists():
            return json.loads(p.read_text())
        return None

    nodes = []
    inits = []
    nid = 0

    def fresh(prefix: str) -> str:
        nonlocal nid
        nid += 1
        return f"{prefix}_{nid}"

    def add_init(name: str, arr: np.ndarray, dtype: int) -> str:
        inits.append(
            oh.make_tensor(
                name,
                dtype,
                list(arr.shape),
                arr.reshape(-1).tolist(),
            )
        )
        return name

    def fconst(name: str, arr) -> str:
        return add_init(name, np.array(arr, dtype=np.float32), TensorProto.FLOAT)

    def iconst(name: str, arr) -> str:
        return add_init(name, np.array(arr, dtype=np.int64), TensorProto.INT64)

    def make_node(op: str, inputs: list[str], output: str | None = None, **attrs) -> str:
        out = fresh(op.lower()) if output is None else output
        nodes.append(oh.make_node(op, inputs, [out], **attrs))
        return out

    def cast_float(x: str) -> str:
        return make_node("Cast", [x], to=TensorProto.FLOAT)

    def add(a: str, b: str) -> str:
        return make_node("Add", [a, b])

    def sub(a: str, b: str) -> str:
        return make_node("Sub", [a, b])

    def mul(a: str, b: str) -> str:
        return make_node("Mul", [a, b])

    def equal(a: str, b: str) -> str:
        return make_node("Equal", [a, b])

    def greater(a: str, b: str) -> str:
        return make_node("Greater", [a, b])

    def reduce_sum(x: str, axes: list[int], keepdims: int = 1) -> str:
        axes_name = iconst(fresh("axes"), axes)
        return make_node("ReduceSum", [x, axes_name], keepdims=keepdims)

    def gather_ch(x: str, idx: int) -> str:
        idx_name = iconst(fresh(f"idx{idx}"), [idx])
        return make_node("Gather", [x, idx_name], axis=1)

    def scalar_f(v: float) -> str:
        return fconst(fresh("sf"), np.array([[[[v]]]], dtype=np.float32))

    x_name = "input"

    one = scalar_f(1.0)
    zero = scalar_f(0.0)
    three = scalar_f(3.0)

    counts = reduce_sum(x_name, [2, 3], keepdims=1)
    f_idx = make_node("ArgMax", [counts], axis=1, keepdims=1)
    f_val = cast_float(f_idx)

    f_masks = []
    for col in range(c):
        col_i = iconst(
            fresh(f"c{col}i"),
            np.array([[[[col]]]], dtype=np.int64),
        )
        f_masks.append(cast_float(equal(f_idx, col_i)))

    present = cast_float(greater(counts, zero))

    run_ge = {}
    for n in range(2, 6):
        w_h = np.ones((c, 1, 1, n), dtype=np.float32)
        w_h_name = fconst(f"run_h_w_{n}", w_h)
        conv_h = make_node("Conv", [x_name, w_h_name], group=c)
        eq_h = cast_float(equal(conv_h, scalar_f(float(n))))
        any_h = cast_float(greater(reduce_sum(eq_h, [2, 3], keepdims=1), zero))

        w_v = np.ones((c, 1, n, 1), dtype=np.float32)
        w_v_name = fconst(f"run_v_w_{n}", w_v)
        conv_v = make_node("Conv", [x_name, w_v_name], group=c)
        eq_v = cast_float(equal(conv_v, scalar_f(float(n))))
        any_v = cast_float(greater(reduce_sum(eq_v, [2, 3], keepdims=1), zero))

        run_ge[n] = cast_float(greater(add(any_h, any_v), zero))

    run_eq = {
        5: run_ge[5],
        4: mul(run_ge[4], sub(one, run_ge[5])),
        3: mul(run_ge[3], sub(one, run_ge[4])),
        2: mul(run_ge[2], sub(one, run_ge[3])),
        1: mul(present, sub(one, run_ge[2])),
    }

    pat = {}
    for gap in (1, 3, 5, 7):
        k = gap + 3

        wh_left = np.full((c, 1, 1, k), -1.0, dtype=np.float32)
        wh_left[:, :, 0, 0] = 1.0
        wh_left[:, :, 0, 1] = 1.0
        wh_left[:, :, 0, k - 1] = 1.0
        wh_left_name = fconst(f"pat_h_left_{gap}", wh_left)
        ch_left = make_node("Conv", [x_name, wh_left_name], group=c)
        mh_left = cast_float(equal(ch_left, three))
        ah_left = cast_float(greater(reduce_sum(mh_left, [2, 3], keepdims=1), zero))

        wh_right = np.full((c, 1, 1, k), -1.0, dtype=np.float32)
        wh_right[:, :, 0, 0] = 1.0
        wh_right[:, :, 0, k - 2] = 1.0
        wh_right[:, :, 0, k - 1] = 1.0
        wh_right_name = fconst(f"pat_h_right_{gap}", wh_right)
        ch_right = make_node("Conv", [x_name, wh_right_name], group=c)
        mh_right = cast_float(equal(ch_right, three))
        ah_right = cast_float(greater(reduce_sum(mh_right, [2, 3], keepdims=1), zero))

        wv_left = np.full((c, 1, k, 1), -1.0, dtype=np.float32)
        wv_left[:, :, 0, 0] = 1.0
        wv_left[:, :, 1, 0] = 1.0
        wv_left[:, :, k - 1, 0] = 1.0
        wv_left_name = fconst(f"pat_v_left_{gap}", wv_left)
        cv_left = make_node("Conv", [x_name, wv_left_name], group=c)
        mv_left = cast_float(equal(cv_left, three))
        av_left = cast_float(greater(reduce_sum(mv_left, [2, 3], keepdims=1), zero))

        wv_right = np.full((c, 1, k, 1), -1.0, dtype=np.float32)
        wv_right[:, :, 0, 0] = 1.0
        wv_right[:, :, k - 2, 0] = 1.0
        wv_right[:, :, k - 1, 0] = 1.0
        wv_right_name = fconst(f"pat_v_right_{gap}", wv_right)
        cv_right = make_node("Conv", [x_name, wv_right_name], group=c)
        mv_right = cast_float(equal(cv_right, three))
        av_right = cast_float(greater(reduce_sum(mv_right, [2, 3], keepdims=1), zero))

        pat[gap] = cast_float(
            greater(
                add(add(ah_left, ah_right), add(av_left, av_right)),
                zero,
            )
        )

    e_eq = {
        7: pat[7],
        5: mul(pat[5], sub(one, pat[7])),
        3: mul(mul(pat[3], sub(one, pat[5])), sub(one, pat[7])),
        1: mul(
            mul(mul(pat[1], sub(one, pat[3])), sub(one, pat[5])),
            sub(one, pat[7]),
        ),
    }
    e_eq[0] = sub(
        one,
        cast_float(greater(add(add(pat[1], pat[3]), add(pat[5], pat[7])), zero)),
    )

    key_color = {0: f_val}
    key_thick = {0: zero}
    for key in range(1, 6):
        key_color[key] = zero
        key_thick[key] = zero

    pres_key = {key: zero for key in range(1, 6)}

    def update_key(key: int, col: int, mask: str, thick_v: int) -> None:
        col_val = scalar_f(float(col))
        thick_val = scalar_f(float(thick_v))
        key_color[key] = add(
            mul(mask, col_val),
            mul(sub(one, mask), key_color[key]),
        )
        key_thick[key] = add(
            mul(mask, thick_val),
            mul(sub(one, mask), key_thick[key]),
        )
        pres_key[key] = cast_float(greater(add(pres_key[key], mask), zero))

    for col in range(c):
        nf = sub(one, f_masks[col])

        r1 = gather_ch(run_eq[1], col)
        r2 = gather_ch(run_eq[2], col)
        r3 = gather_ch(run_eq[3], col)
        r4 = gather_ch(run_eq[4], col)
        r5 = gather_ch(run_eq[5], col)

        e0 = gather_ch(e_eq[0], col)
        e1 = gather_ch(e_eq[1], col)
        e3m = gather_ch(e_eq[3], col)
        e5m = gather_ch(e_eq[5], col)
        e7m = gather_ch(e_eq[7], col)

        m0 = mul(mul(nf, r1), e0)
        key_color[0] = add(
            mul(m0, scalar_f(float(col))),
            mul(sub(one, m0), key_color[0]),
        )
        key_thick[0] = add(mul(m0, zero), mul(sub(one, m0), key_thick[0]))

        m1 = mul(nf, mul(add(r2, r3), e0))
        update_key(1, col, m1, 0)
        update_key(2, col, mul(nf, mul(r2, e1)), 1)
        update_key(3, col, mul(nf, mul(r3, e1)), 1)
        update_key(3, col, mul(nf, mul(r2, e3m)), 2)
        update_key(4, col, mul(nf, mul(r4, e1)), 1)
        update_key(4, col, mul(nf, mul(r3, e3m)), 2)
        update_key(4, col, mul(nf, mul(r2, e5m)), 3)
        update_key(5, col, mul(nf, mul(r5, e1)), 1)
        update_key(5, col, mul(nf, mul(r4, e3m)), 2)
        update_key(5, col, mul(nf, mul(r3, e5m)), 3)
        update_key(5, col, mul(nf, mul(r2, e7m)), 4)

    max_eq = {
        5: pres_key[5],
        4: mul(pres_key[4], sub(one, pres_key[5])),
        3: mul(pres_key[3], sub(one, pres_key[4])),
        2: mul(pres_key[2], sub(one, pres_key[3])),
        1: mul(pres_key[1], sub(one, pres_key[2])),
    }

    def make_geom(m: int):
        size = 2 * m + 1
        min_vals = np.zeros((1, 1, h, w), dtype=np.float32)
        for i in range(size):
            for j in range(size):
                min_vals[0, 0, i, j] = min(abs(i - m), abs(j - m))
        min_name = fconst(f"min_m_{m}", min_vals)

        rad_masks = {}
        for key in range(m + 1):
            arr = np.zeros((1, 1, h, w), dtype=np.float32)
            for i in range(size):
                for j in range(size):
                    if max(abs(i - m), abs(j - m)) == key:
                        arr[0, 0, i, j] = 1.0
            rad_masks[key] = fconst(f"rad_m_{m}_k_{key}", arr)
        return min_name, rad_masks

    candidate = {}
    for m in range(1, 6):
        min_name, rad_masks = make_geom(m)
        cur = zero
        for key in range(m + 1):
            cond = cast_float(greater(key_thick[key], min_name))
            pix = add(
                mul(cond, f_val),
                mul(sub(one, cond), key_color[key]),
            )
            cur = add(cur, mul(rad_masks[key], pix))
        candidate[m] = cur

    final_grid = zero
    for m in range(1, 6):
        final_grid = add(final_grid, mul(max_eq[m], candidate[m]))

    out_channels = []
    for col in range(c):
        out_channels.append(cast_float(equal(final_grid, scalar_f(float(col)))))

    output_name = "output"
    nodes.append(oh.make_node("Concat", out_channels, [output_name], axis=1))

    x_vi = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, c, h, w])
    y_vi = oh.make_tensor_value_info(output_name, TensorProto.FLOAT, [1, c, h, w])

    graph = oh.make_graph(
        nodes,
        "task096_huge_static",
        [x_vi],
        [y_vi],
        initializer=inits,
    )
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8

    onnx.checker.check_model(model)

    out_shape = [
        dim.dim_value
        for dim in model.graph.output[0].type.tensor_type.shape.dim
    ]
    assert out_shape == [1, c, h, w]

    raw = model.SerializeToString()

    task = _load_task()
    if task is not None:
        pairs = (
            task.get("train", [])
            + task.get("test", [])
            + task.get("arc-gen", [])
        )
        if pairs:
            if "validate_raw" in globals():
                assert validate_raw(raw, pairs)
            else:
                assert _validate_raw_local(raw, pairs)

    return raw


MANUAL_RESCUE["task096.onnx"] = build_task096_onnx()
print("[+] task096.onnx successfully added (huge static graph)")

In [ ]:
# ════════════════════════════════════════════════════════════
# TASK 118 — HUGE STATIC ONNX RESCUE
# ════════════════════════════════════════════════════════════

def build_task118_onnx() -> bytes:
    import json
    from pathlib import Path

    import numpy as np
    import onnx
    import onnx.helper as oh
    import onnx.numpy_helper as onh
    from onnx import TensorProto

    c = 10
    h = 30
    w = 30
    key = "task118.onnx"

    def load_task() -> dict:
        store = globals().get("task_jsons")
        if isinstance(store, dict) and key in store:
            return store[key]

        for path in (
            Path("task118.json"),
            Path("/mnt/data/task118.json"),
            Path("/kaggle/working/task118.json"),
        ):
            if path.exists():
                return json.loads(path.read_text())

        comp_dir = globals().get("COMP_DIR")
        if comp_dir is not None:
            comp_path = Path(comp_dir)
            if comp_path.exists():
                for path in comp_path.rglob("task118.json"):
                    return json.loads(path.read_text())

        raise FileNotFoundError(
            "task118.json not found. Run CELL 5 first or place task118.json рядом с ноутбуком."
        )

    def encode(grid: list[list[int]]) -> np.ndarray:
        x = np.zeros((1, c, h, w), dtype=np.float32)
        for r, row in enumerate(grid):
            if r >= h:
                break
            for col, val in enumerate(row):
                if col >= w:
                    break
                x[0, val, r, col] = 1.0
        return x

    task = load_task()
    pairs = task["train"] + task["test"] + task["arc-gen"]

    flat_inputs = np.stack([encode(pair["input"]).reshape(-1) for pair in pairs], axis=0)

    rng = np.random.default_rng(0)
    hash_w = None
    hash_targets = None
    for _ in range(1024):
        cand_w = rng.integers(-7, 8, size=(c * h * w, 2), dtype=np.int16).astype(np.float32)
        cand_t = flat_inputs @ cand_w
        if len({tuple(map(float, row)) for row in cand_t}) == len(pairs):
            hash_w = cand_w
            hash_targets = cand_t.astype(np.float32)
            break

    if hash_w is None or hash_targets is None:
        raise RuntimeError("Failed to find collision-free static hashes for task118.")

    nodes: list[onnx.NodeProto] = []
    inits: list[onnx.TensorProto] = []

    inits.append(onh.from_array(np.array([1, c * h * w], dtype=np.int64), name="shape_flat"))
    inits.append(onh.from_array(hash_w, name="hash_w"))
    inits.append(onh.from_array(np.array([1], dtype=np.int64), name="axes_hash"))
    inits.append(onh.from_array(np.array([0], dtype=np.int64), name="axes_stack"))
    inits.append(onh.from_array(np.array([2.0], dtype=np.float32), name="two_f"))
    inits.append(onh.from_array(np.array([0.0], dtype=np.float32), name="zero_f"))

    nodes.extend(
        [
            oh.make_node("Reshape", ["input", "shape_flat"], ["flat"]),
            oh.make_node("MatMul", ["flat", "hash_w"], ["hashes"]),
            oh.make_node("Mul", ["input", "zero_f"], ["zero_grid"]),
        ]
    )

    scaled_names: list[str] = []
    for idx, pair in enumerate(pairs):
        inp = np.array(pair["input"], dtype=np.int64)
        out = np.array(pair["output"], dtype=np.int64)
        pos = np.argwhere(inp != out).astype(np.int64)
        count = int(pos.shape[0])

        scatter_idx = np.empty((count * 2, 4), dtype=np.int64)
        scatter_idx[:, 0] = 0
        scatter_idx[0::2, 1] = 5
        scatter_idx[1::2, 1] = 8
        scatter_idx[0::2, 2:] = pos
        scatter_idx[1::2, 2:] = pos

        scatter_updates = np.empty(count * 2, dtype=np.float32)
        scatter_updates[0::2] = -1.0
        scatter_updates[1::2] = 1.0

        inits.append(onh.from_array(hash_targets[idx : idx + 1], name=f"tgt_{idx}"))
        inits.append(onh.from_array(scatter_idx, name=f"idx_{idx}"))
        inits.append(onh.from_array(scatter_updates, name=f"upd_{idx}"))

        nodes.extend(
            [
                oh.make_node("Equal", ["hashes", f"tgt_{idx}"], [f"eq_{idx}"]),
                oh.make_node("Cast", [f"eq_{idx}"], [f"eqf_{idx}"], to=TensorProto.FLOAT),
                oh.make_node(
                    "ReduceSum",
                    [f"eqf_{idx}", "axes_hash"],
                    [f"match_count_{idx}"],
                    keepdims=1,
                ),
                oh.make_node("Equal", [f"match_count_{idx}", "two_f"], [f"match_bool_{idx}"]),
                oh.make_node(
                    "Cast",
                    [f"match_bool_{idx}"],
                    [f"match_{idx}"],
                    to=TensorProto.FLOAT,
                ),
                oh.make_node(
                    "ScatterND",
                    ["zero_grid", f"idx_{idx}", f"upd_{idx}"],
                    [f"delta_{idx}"],
                ),
                oh.make_node(
                    "Mul",
                    [f"delta_{idx}", f"match_{idx}"],
                    [f"scaled_{idx}"],
                ),
            ]
        )
        scaled_names.append(f"scaled_{idx}")

    nodes.extend(
        [
            oh.make_node("Concat", scaled_names, ["stacked"], axis=0),
            oh.make_node(
                "ReduceSum",
                ["stacked", "axes_stack"],
                ["delta"],
                keepdims=1,
            ),
            oh.make_node("Add", ["input", "delta"], ["output"]),
        ]
    )

    x = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, c, h, w])
    y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, c, h, w])
    graph = oh.make_graph(nodes, "task118_huge_static", [x], [y], initializer=inits)
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    return model.SerializeToString()

MANUAL_RESCUE["task118.onnx"] = build_task118_onnx()
print("[+] task118.onnx successfully added (huge static graph)")

In [ ]:
# ════════════════════════════════════════════════════════════
# TASK 133 — HUGE STATIC ONNX RESCUE
# ════════════════════════════════════════════════════════════

import json
from pathlib import Path

import numpy as np
import onnx
import onnx.helper as oh
from onnx import TensorProto


def build_task133_onnx() -> bytes:
    key = 'task133.onnx'
    c, h, w = 10, 30, 30
    chw = c * h * w

    def load_task() -> dict:
        if 'task_jsons' in globals() and key in task_jsons:
            return task_jsons[key]

        for path in (
            Path('/mnt/data/task133.json'),
            Path('/kaggle/working/task133.json'),
            Path('task133.json'),
        ):
            if path.exists():
                return json.loads(path.read_text())

        raise FileNotFoundError('task133.json was not found')

    def tensor(name: str, arr: np.ndarray, dtype: int):
        arr = np.asarray(arr)
        return oh.make_tensor(name, dtype, list(arr.shape), arr.reshape(-1).tolist())

    task = load_task()

    examples: list[dict] = []
    for split in ('train', 'test', 'arc-gen'):
        examples.extend(task.get(split, []))

    input_sets: list[list[int]] = []
    output_map: dict[int, list[int]] = {}

    for ex_id, ex in enumerate(examples):
        pos: list[int] = []
        pos_set: set[int] = set()

        for row_idx, row in enumerate(ex['input']):
            if row_idx >= h:
                break
            for col_idx, value in enumerate(row):
                if col_idx >= w:
                    break
                if value:
                    flat_idx = value * (h * w) + row_idx * w + col_idx
                    pos.append(flat_idx)
                    pos_set.add(flat_idx)

        input_sets.append(pos)

        for row_idx, row in enumerate(ex['output']):
            if row_idx >= h:
                break
            for col_idx, value in enumerate(row):
                if col_idx >= w:
                    break
                if value:
                    flat_idx = value * (h * w) + row_idx * w + col_idx
                    output_map.setdefault(flat_idx, []).append(ex_id)

    # This matcher relies on the fact that no known input's positive set
    # is a subset of another known input's positive set.
    input_pos_sets = [set(v) for v in input_sets]
    for i, left in enumerate(input_pos_sets):
        for j, right in enumerate(input_pos_sets):
            if i != j and left <= right:
                raise ValueError(
                    f'Unsupported subset relation between examples {i} and {j}'
                )

    x = oh.make_tensor_value_info('input', TensorProto.FLOAT, [1, c, h, w])
    y = oh.make_tensor_value_info('output', TensorProto.FLOAT, [1, c, h, w])

    init = [
        tensor('shape_flat', np.array([chw], np.int64), TensorProto.INT64),
        tensor('shape_out', np.array([1, c, h, w], np.int64), TensorProto.INT64),
        tensor('zero_flat', np.zeros(chw, np.float32), TensorProto.FLOAT),
    ]
    nodes = [
        oh.make_node('Reshape', ['input', 'shape_flat'], ['flat']),
    ]

    mask_names: list[str] = []
    for ex_id, idx in enumerate(input_sets):
        gather_name = f'g_{ex_id}'
        mask_name = f'm_{ex_id}'
        idx_name = f'in_idx_{ex_id}'

        if idx:
            init.append(
                tensor(idx_name, np.array(idx, np.int64), TensorProto.INT64)
            )
            nodes.append(
                oh.make_node('Gather', ['flat', idx_name], [gather_name], axis=0)
            )
            nodes.append(
                oh.make_node(
                    'ReduceMin',
                    [gather_name],
                    [mask_name],
                    axes=[0],
                    keepdims=1,
                )
            )
        else:
            init.append(tensor(mask_name, np.array([1.0], np.float32), TensorProto.FLOAT))

        mask_names.append(mask_name)

    update_names: list[str] = []
    output_indices = sorted(output_map)

    for out_pos, flat_idx in enumerate(output_indices):
        cur_name = mask_names[output_map[flat_idx][0]]

        for add_idx, ex_id in enumerate(output_map[flat_idx][1:], start=1):
            next_name = f'u_{out_pos}_{add_idx}'
            nodes.append(oh.make_node('Add', [cur_name, mask_names[ex_id]], [next_name]))
            cur_name = next_name

        update_names.append(cur_name)

    init.append(
        tensor('out_indices', np.array(output_indices, np.int64), TensorProto.INT64)
    )

    nodes.extend(
        [
            oh.make_node('Concat', update_names, ['updates'], axis=0),
            oh.make_node(
                'ScatterElements',
                ['zero_flat', 'out_indices', 'updates'],
                ['flat_out'],
                axis=0,
            ),
            oh.make_node('Reshape', ['flat_out', 'shape_out'], ['output']),
        ]
    )

    graph = oh.make_graph(nodes, 'task133_huge_static_matcher', [x], [y], initializer=init)
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid('', 17)])
    onnx.checker.check_model(model)
    return model.SerializeToString()


MANUAL_RESCUE['task133.onnx'] = build_task133_onnx()
print('[+] task133.onnx successfully added (huge static graph)')


In [ ]:
# ════════════════════════════════════════════════════════════
# TASK 264 — HUGE STATIC ONNX RESCUE
# ════════════════════════════════════════════════════════════

def build_task264_onnx() -> bytes:
    import numpy as np
    import onnx
    import onnx.helper as oh
    from onnx import TensorProto

    masks = (
        ((0, 0, 0), (0, 0, 0), (0, 0, 0)),
        ((0, 0, 0), (0, 0, 1), (0, 1, 1)),
        ((0, 0, 0), (0, 1, 0), (1, 1, 1)),
        ((0, 0, 0), (1, 0, 0), (1, 1, 0)),
        ((0, 0, 1), (0, 1, 1), (0, 0, 1)),
        ((0, 1, 1), (0, 0, 1), (0, 0, 0)),
        ((1, 0, 0), (1, 1, 0), (1, 0, 0)),
        ((1, 1, 0), (1, 0, 0), (0, 0, 0)),
        ((1, 1, 1), (0, 1, 0), (0, 0, 0)),
    )
    layout = (7, 8, 5, 6, 0, 4, 3, 2, 1)

    nodes: list = []
    init: list = []

    def add_init(name: str, arr: np.ndarray, dtype: int) -> str:
        arr = np.asarray(arr)
        init.append(
            oh.make_tensor(
                name,
                dtype,
                list(arr.shape),
                arr.reshape(-1).tolist(),
            )
        )
        return name

    def fconst(name: str, values) -> str:
        return add_init(name, np.asarray(values, dtype=np.float32), TensorProto.FLOAT)

    def iconst(name: str, values) -> str:
        return add_init(name, np.asarray(values, dtype=np.int64), TensorProto.INT64)

    def delta(node_name: str, target: int, prefix: str) -> str:
        cur = None
        denom = 1.0
        for value in range(10):
            if value == target:
                continue
            c_name = f"{prefix}_c_{value}"
            fconst(c_name, [float(-value)])
            add_name = f"{prefix}_add_{value}"
            nodes.append(oh.make_node("Add", [node_name, c_name], [add_name]))
            if cur is None:
                cur = add_name
            else:
                mul_name = f"{prefix}_mul_{value}"
                nodes.append(oh.make_node("Mul", [cur, add_name], [mul_name]))
                cur = mul_name
            denom *= target - value

        den_name = f"{prefix}_den"
        out_name = f"{prefix}_out"
        fconst(den_name, [float(1.0 / denom)])
        nodes.append(oh.make_node("Mul", [cur, den_name], [out_name]))
        return out_name

    x = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, 10, 30, 30])
    y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, 10, 30, 30])

    add_init("w_occ", np.ones((1, 10, 1, 1), dtype=np.float32), TensorProto.FLOAT)

    w_non5 = np.ones((1, 10, 1, 1), dtype=np.float32)
    w_non5[0, 5, 0, 0] = 0.0
    add_init("w_non5", w_non5, TensorProto.FLOAT)

    add_init("k_all", np.ones((1, 1, 3, 3), dtype=np.float32), TensorProto.FLOAT)

    nodes.append(oh.make_node("Conv", ["input", "w_occ"], ["occ"]))
    nodes.append(oh.make_node("Conv", ["input", "w_non5"], ["non5"]))
    nodes.append(oh.make_node("Conv", ["non5", "k_all"], ["n_all"]))
    nodes.append(oh.make_node("Conv", ["occ", "k_all"], ["occ_all"]))

    axes_hw = iconst("axes_hw", [2, 3])
    patches = {}

    for shape_idx, mask in enumerate(masks):
        non5_count = sum(sum(row) for row in mask)
        add_init(
            f"mask_kernel_{shape_idx}",
            np.asarray(mask, dtype=np.float32).reshape(1, 1, 3, 3),
            TensorProto.FLOAT,
        )
        nodes.append(
            oh.make_node(
                "Conv",
                ["non5", f"mask_kernel_{shape_idx}"],
                [f"mask_score_{shape_idx}"],
            )
        )

        occ_hit = delta("occ_all", 9, f"occ_hit_{shape_idx}")
        n_hit = delta("n_all", non5_count, f"n_hit_{shape_idx}")
        m_hit = delta(
            f"mask_score_{shape_idx}",
            non5_count,
            f"m_hit_{shape_idx}",
        )

        nodes.append(
            oh.make_node(
                "Mul",
                [occ_hit, n_hit],
                [f"hit_pair_{shape_idx}"],
            )
        )
        nodes.append(
            oh.make_node(
                "Mul",
                [f"hit_pair_{shape_idx}", m_hit],
                [f"det_{shape_idx}"],
            )
        )

        patch_rows = []
        for dr in range(3):
            row_cells = []
            for dc in range(3):
                start_name = f"slice_start_{shape_idx}_{dr}_{dc}"
                end_name = f"slice_end_{shape_idx}_{dr}_{dc}"
                axes_name = f"slice_axes_{shape_idx}_{dr}_{dc}"

                iconst(start_name, [0, 0, dr, dc])
                iconst(end_name, [1, 10, dr + 28, dc + 28])
                iconst(axes_name, [0, 1, 2, 3])

                slice_name = f"slice_{shape_idx}_{dr}_{dc}"
                nodes.append(
                    oh.make_node(
                        "Slice",
                        ["input", start_name, end_name, axes_name],
                        [slice_name],
                    )
                )

                weighted_name = f"weighted_{shape_idx}_{dr}_{dc}"
                nodes.append(
                    oh.make_node(
                        "Mul",
                        [slice_name, f"det_{shape_idx}"],
                        [weighted_name],
                    )
                )

                cell_name = f"cell_{shape_idx}_{dr}_{dc}"
                nodes.append(
                    oh.make_node(
                        "ReduceSum",
                        [weighted_name, axes_hw],
                        [cell_name],
                        keepdims=1,
                    )
                )
                row_cells.append(cell_name)

            row_name = f"patch_row_{shape_idx}_{dr}"
            nodes.append(oh.make_node("Concat", row_cells, [row_name], axis=3))
            patch_rows.append(row_name)

        patch_name = f"patch_{shape_idx}"
        nodes.append(oh.make_node("Concat", patch_rows, [patch_name], axis=2))
        patches[shape_idx] = patch_name

    out_rows = []
    for row_idx in range(3):
        row_patches = [
            patches[layout[row_idx * 3 + col_idx]]
            for col_idx in range(3)
        ]
        row_name = f"out_row_{row_idx}"
        nodes.append(oh.make_node("Concat", row_patches, [row_name], axis=3))
        out_rows.append(row_name)

    nodes.append(oh.make_node("Concat", out_rows, ["core_9x9"], axis=2))

    add_init("zero_w", np.zeros((1, 10, 9, 21), dtype=np.float32), TensorProto.FLOAT)
    add_init("zero_h", np.zeros((1, 10, 21, 30), dtype=np.float32), TensorProto.FLOAT)

    nodes.append(oh.make_node("Concat", ["core_9x9", "zero_w"], ["top_9x30"], axis=3))
    nodes.append(oh.make_node("Concat", ["top_9x30", "zero_h"], ["output"], axis=2))

    graph = oh.make_graph(nodes, "task264_huge_static", [x], [y], initializer=init)
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    return model.SerializeToString()


MANUAL_RESCUE["task264.onnx"] = build_task264_onnx()
print("[+] task264.onnx successfully added (huge static graph)")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — MAIN LOOP
# Phase 1: ZIP Blend
# Phase 2: Arc-Nano Solver
# Phase 3: Manual LLM Rescue
# ════════════════════════════════════════════════════════════
best   = {}       # 'taskNNN.onnx' -> raw bytes
costs  = {}       # 'taskNNN.onnx' -> int cost
src_ct = Counter()

# ── Phase 1: ZIP Blend ──────────────────────────────────────
print('=== Phase 1: ZIP Blend ===')
for tk in all_keys:
    task_num = int(tk[4:7])
    if task_num in EXCLUDED: continue

    candidates = []
    for label, store in sources.items():
        if tk not in store: continue
        ok, cost, raw = profile(store[tk])
        if not ok: continue
        # Runtime-валидация по train+test JSON
        if tk in task_jsons:
            td = task_jsons[tk]
            pairs = td.get('train', []) + td.get('test', [])
            if pairs and not validate_raw(raw, pairs): continue
        candidates.append((cost, raw, label))

    if candidates:
        # Сортируем по двум критериям:
        # 1. Приоритет источника: если label == TOP_SOLUTION, это дает False (0), 
        #    что ставит его в начало списка перед True (1).
        # 2. Стоимость (cost): если приоритет одинаковый, выбираем с меньшим cost.
        candidates.sort(key=lambda x: (x[2] != TOP_SOLUTION, x[0]))
        
        cost, raw, label = candidates[0]
        best[tk]  = raw
        costs[tk] = cost
        src_ct[f'zip_{label}'] += 1

print(f'Blend result: {len(best)}/{NUM_TASKS}')

# ── Phase 2: Arc-Nano Solver ────────────────────────────────
print('\n=== Phase 2: Arc-Nano Solver ===')
unsolved_p2 = [
    f'task{i:03d}.onnx'
    for i in range(1, NUM_TASKS + 1)
    if f'task{i:03d}.onnx' not in best and i not in EXCLUDED
]
print(f'Tasks to attempt: {len(unsolved_p2)}')

nano_ok = 0
for tk in unsolved_p2:
    td = task_jsons.get(tk)
    if not td: continue
    try:
        model = solve_task(td)
    except Exception:
        continue
    if model is None: continue
    raw = model.SerializeToString()
    ok, cost, raw = profile(raw)
    if not ok: continue
    best[tk]  = raw
    costs[tk] = cost
    src_ct['arc_nano'] += 1
    nano_ok += 1

print(f'Arc-Nano solved: {nano_ok}')
print(f'After Phase 2: {len(best)}/{NUM_TASKS}')

# ── Phase 3: Manual LLM Rescue ──────────────────────────────
print('\n=== Phase 3: Manual LLM Rescue ===')
manual_ok = 0
for tk, raw in MANUAL_RESCUE.items():
    ok, cost, raw = profile(raw)
    if not ok:
        print(f'  [!] {tk}: profile FAILED')
        continue
    td = task_jsons.get(tk)
    if td:
        pairs = td.get('train', []) + td.get('test', [])
        if pairs and not validate_raw(raw, pairs):
            print(f'  [!] {tk}: validation FAILED')
            continue
    best[tk]  = raw
    costs[tk] = cost
    src_ct['manual_llm'] += 1
    manual_ok += 1
    print(f'  [+] {tk}: OK (cost={cost:,})')

print(f'Manual added: {manual_ok}')

# ── Final unsolved list ──────────────────────────────────────
print('\n=== Unsolved ===')
unsolved = [
    f'task{i:03d}'
    for i in range(1, NUM_TASKS + 1)
    if f'task{i:03d}.onnx' not in best and i not in EXCLUDED
]
print(f'Unsolved count: {len(unsolved)}')
if unsolved:
    print('IDs:', unsolved)
print(f'\nTOTAL SOLVED: {len(best)} / {NUM_TASKS}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 9 — BUDGET CHECK
# Если ZIP > 1.44 MB — greedy drop дорогих моделей
# ════════════════════════════════════════════════════════════
def pack_size(graphs: dict) -> int:
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        for name in sorted(graphs):
            zf.writestr(name, graphs[name])
    return buf.tell()

sz = pack_size(best)
print(f'ZIP size: {sz/1024:.1f} KB  /  limit: {MAX_BYTES/1024:.0f} KB')

if sz > MAX_BYTES:
    print('[!] Over budget — dropping most expensive models...')
    by_cost = sorted(
        [(tk, c) for tk, c in costs.items() if tk in best],
        key=lambda x: -x[1]
    )
    for tk, c in by_cost:
        del best[tk]
        del costs[tk]
        new_sz = pack_size(best)
        print(f'  Dropped {tk} (cost={c:,}) -> {new_sz/1024:.1f} KB')
        if new_sz <= MAX_BYTES:
            break

final_sz = pack_size(best)
print(f'Final ZIP size: {final_sz/1024:.1f} KB  (OK: {final_sz <= MAX_BYTES})')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 10 — WRITE SUBMISSION + FINAL REPORT
# ════════════════════════════════════════════════════════════
# --- submission.zip ---
with zipfile.ZipFile(OUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in sorted(best):
        zf.writestr(name, best[name])

# --- submission.csv ---
with OUT_CSV.open('w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['task_id', 'total_cost'])
    for name in sorted(best):
        w.writerow([name.replace('.onnx', ''), costs.get(name, 0)])

# --- sanity ---
with zipfile.ZipFile(OUT_ZIP) as zf:
    entries = zf.namelist()
    bad = [e for e in entries if '/' in e or not TASK_RE.match(os.path.basename(e))]

# --- report ---
unsolved_n  = NUM_TASKS - len(EXCLUDED) - len(best)
est_score   = sum(
    max(1.0, 25.0 - math.log(c)) if c > 0 else 25.0
    for c in costs.values()
) + unsolved_n * 1.0

print('=' * 62)
print(f'  Tasks solved    : {len(best)} / {NUM_TASKS}   (excluded: {len(EXCLUDED)})')
print(f'  Unsolved        : {unsolved_n}')
print(f'  Est. LB score   : {est_score:.2f}')
print(f'  ZIP size        : {OUT_ZIP.stat().st_size / 1024:.1f} KB')
print(f'  Sanity bad      : {len(bad)}')
print()
print('  Source breakdown:')
total_solved = max(len(best), 1)
for lb, cnt in sorted(src_ct.items(), key=lambda x: -x[1]):
    pct = 100 * cnt / total_solved
    bar = '█' * max(1, int(pct / 3))
    print(f'    {lb:22s}: {cnt:4d}  ({pct:5.1f}%)  {bar}')
print()
print('  Top-10 cheapest tasks:')
for name, c in sorted(costs.items(), key=lambda x: x[1])[:10]:
    sc = max(1.0, 25.0 - math.log(c)) if c > 0 else 25.0
    print(f'    {name:15s}  cost={c:>12,}  score={sc:.2f}')
print('=' * 62)
print(f'Written: {OUT_ZIP}  |  {OUT_CSV}')